# التجربة الثالثة: ضوضاء واقعية + "شخّص ثم عالج"

**ضع في المجلد نفسه:** هذا الدفتر + `overlap_experiment.py` + `experiment_v2.py` + `experiment_v3.py` + `analyze_v3.py`.

اقرأ `PROTOCOL_v3.md` أولاً: فيه الفرضيات الجديدة والاختبارات الأساسية، مثبّتة قبل التشغيل.

قبل التشغيل: الشاحن موصول، ووضع Performance/Turbo، والسكون معطّل.

In [ ]:
import os, sys, argparse, importlib
from pathlib import Path
for f in ["overlap_experiment.py", "experiment_v2.py", "experiment_v3.py", "analyze_v3.py"]:
    assert Path(f).exists(), f"ضع {f} في نفس مجلد الدفتر"
sys.path.insert(0, os.getcwd())
import overlap_experiment, experiment_v2, experiment_v3 as v3
for m in (overlap_experiment, experiment_v2, v3): importlib.reload(m)
from IPython.display import Markdown, Image, display
print("Logical CPUs:", os.cpu_count())

def args_v3(source="openml", preset="full", max_datasets=20, seeds=3, n_jobs=10,
            fracs=(0.02, 0.05, 0.10), max_rows=3000, max_features=200, out="results_v3"):
    return argparse.Namespace(source=source, preset=preset, max_datasets=max_datasets, seeds=seeds,
                              n_jobs=n_jobs, fracs=list(fracs), max_rows=max_rows,
                              max_features=max_features, out=out)

def show_v3(out):
    display(Markdown(Path(out, "summary_v3.md").read_text(encoding="utf-8")))
    for f in ["fig_v3_strategies.png", "fig_v3_diagnostics.png"]:
        if Path(out, f).exists():
            display(Image(filename=str(Path(out, f)), width=700))

## ١. اختبار سريع دون إنترنت (2 إلى 3 دقائق)

In [ ]:
v3.run(args_v3(source="sklearn", preset="smoke", max_datasets=2, seeds=2, n_jobs=4,
               fracs=(0.05, 0.10), out="v3_smoke"))
%run analyze_v3.py v3_smoke
show_v3("v3_smoke")

## ٢. التجربة الأولية: 20 مجموعة × 3 بذور × 11 ظرفاً
**الوقت المتوقع على جهازك: نحو 3 إلى 4 ساعات.** شغّلها ليلاً.

للإيقاف اضغط ■، وللاستئناف أعد تشغيل الخلية نفسها.

In [ ]:
v3.run(args_v3(source="openml", preset="full", max_datasets=20, seeds=3, n_jobs=10, out="results_v3"))

## ٣. التحليل

In [ ]:
%run analyze_v3.py results_v3
show_v3("results_v3")

## ٤. تحليل حسب النموذج على نتائج التجربة الأولية (ثوانٍ، دون حساب جديد)

In [ ]:
%run analyze_v3_confirm.py results_v3

## ٥. التجربة الكاملة التأكيدية: 40 مجموعة × 5 بذور
اقرأ `PROTOCOL_v3_confirm.md` أولاً.

- **الوقت المتوقع: نحو 8 إلى 12 ساعة.** شغّلها ليلاً، والجهاز على الشاحن، والسكون معطّل.
- المهام الكبيرة تبدأ أولاً، فلا تتأخر النهاية بسببها.
- للإيقاف اضغط ■، وللاستئناف أعد تشغيل الخلية نفسها.
- **لا تعدّل أي ملف بعد بدء التشغيل.**

In [ ]:
v3.run(args_v3(source="openml", preset="full", max_datasets=40, seeds=5,
               max_rows=2000, n_jobs=10, out="results_v3_full"))

## ٦. التحليل الكامل + الاختبار التأكيدي

In [ ]:
%run analyze_v3.py results_v3_full
show_v3("results_v3_full")
%run analyze_v3_confirm.py results_v3_full --discovery-from results_v3